In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

In [14]:
portfolio = pd.read_csv("/Users/fuyuxuan/Downloads/test9_1_portfolio.csv")
returns = pd.read_csv("/Users/fuyuxuan/Downloads/test9_1_returns.csv")

symbols = portfolio["Stock"].values
shares = portfolio["Holding"].astype(float).values
p0 = portfolio["Starting Price"].astype(float).values

values = shares * p0
total_value = values.sum()

rA = returns[symbols[0]].values
rB = returns[symbols[1]].values

# Fit marginals based on the "Distribution" column
# A = Normal, B = T
muA, sigmaA = stats.norm.fit(rA)              # MLE
dfB, locB, scaleB = stats.t.fit(rB)           # MLE

# Copula dependence (Gaussian copula)
# Use Kendall tau -> rho transform
tau, _ = stats.kendalltau(rA, rB)
rho = np.sin(np.pi / 2 * tau)


# Simulate correlated uniforms
np.random.seed(0)
n_sim = 100000

z = np.random.multivariate_normal([0, 0], [[1, rho], [rho, 1]], size=n_sim)
u = stats.norm.cdf(z)

# Invert marginals
rA_sim = stats.norm.ppf(u[:, 0], loc=muA, scale=sigmaA)
rB_sim = stats.t.ppf(u[:, 1], dfB, loc=locB, scale=scaleB)

# Convert returns -> dollar losses
lossA = -(values[0] * rA_sim)
lossB = -(values[1] * rB_sim)
lossT = lossA + lossB

def var_es(losses, alpha=0.95):
    v = np.quantile(losses, alpha)
    e = losses[losses >= v].mean()
    return v, e

VaRA, ESA = var_es(lossA)
VaRB, ESB = var_es(lossB)
VaRT, EST = var_es(lossT)

# Percent columns must be derived from dollar risk / value
result = pd.DataFrame({
    "Stock": [symbols[0], symbols[1], "Total"],
    "VaR95": [VaRA, VaRB, VaRT],
    "ES95": [ESA, ESB, EST],
    "VaR95_Pct": [VaRA / values[0], VaRB / values[1], VaRT / total_value],
    "ES95_Pct": [ESA / values[0], ESB / values[1], EST / total_value],
})
print(result)

   Stock       VaR95        ES95  VaR95_Pct  ES95_Pct
0      A   94.138301  117.740170   0.047069  0.058870
1      B  107.432026  151.453770   0.035811  0.050485
2  Total  152.464101  200.781215   0.030493  0.040156
